# Claim Type Classifier v2: Fine-tuning MuRIL with LoRA (English + Hindi + Telugu)
**Project:** Multimodal Motor Insurance Claim Agent (ShaikSure Motor Insurance, fictional)

**Goal:** read a customer's claim description in **English, Hindi, Telugu, Hinglish or Tenglish** and predict the
claim type: `collision`, `theft`, `flood`, `fire`, `vandalism` or `glass_only`.

**Why MuRIL:** v1 used English DistilBERT and failed on Hindi words like *gaadi* and *paani*.
MuRIL (Google) is a BERT model pre-trained on 17 Indian languages, including romanized text like "gaadi chori ho gayi".

**Steps**
1. Load the synthetic labelled data from GitHub
2. Split it into train / validation / test (stratified)
3. Add LoRA adapters to MuRIL and fine-tune only those small parts
4. Evaluate: test set, English hard set, multilingual hard set, per-language scores
5. Compare with zero-shot GPT-4o-mini (accuracy, speed, cost)
6. Save the LoRA adapter and download the results

**Before running:** Runtime > Change runtime type > **T4 GPU**. Add the `OPENAI_API_KEY` secret (key icon). Then **Run all**.

In [ ]:
# Step 0: install the two libraries Colab does not have by default
!pip install -q peft openai
# Remove Colab's old torchao: peft refuses to load with it, and we don't use it
!pip uninstall -y -q torchao

In [ ]:
# Step 1: imports and settings
import json
import os
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, Trainer,
                          TrainingArguments, set_seed)

MODEL_NAME = "google/muril-base-cased"   # BERT model for 17 Indian languages (Google)
MAX_LENGTH = 128                          # claim descriptions are short, 128 tokens is enough
EPOCHS = 10
SEED = 42                                 # same seed = same results every run
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

In [ ]:
# Step 2: load the data straight from the public GitHub repo (no manual upload needed)
REPO = "https://raw.githubusercontent.com/IBRAHIM-1245/multimodal-motor-insurance-claim-agent/main"
data = pd.read_csv(f"{REPO}/data/claims/claim_descriptions.csv")
hard_en = pd.read_csv(f"{REPO}/data/claims/hard_test_claims.csv")
hard_multi = pd.read_csv(f"{REPO}/data/claims/hard_test_multilingual.csv")

print("Training data:", data.shape, "| English hard set:", hard_en.shape, "| Multilingual hard set:", hard_multi.shape)
print(pd.crosstab(data["label"], data["language"]))
data.sample(5, random_state=1)

In [ ]:
# Step 3: turn label names into numbers (models work with numbers)
LABELS = ["collision", "theft", "flood", "fire", "vandalism", "glass_only"]
label2id = {name: i for i, name in enumerate(LABELS)}
id2label = {i: name for i, name in enumerate(LABELS)}

for df in (data, hard_en, hard_multi):
    df["label_id"] = df["label"].map(label2id)

# Split 80% train, 10% validation, 10% test.
# stratify = keep the same share of each claim type in every split.
train_df, temp_df = train_test_split(data, test_size=0.2, stratify=data["label_id"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=SEED)
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

In [ ]:
# Step 4: tokenize. The tokenizer splits text into sub-word tokens and turns them into ids.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ClaimDataset(torch.utils.data.Dataset):
    """Holds tokenized claim texts and their label ids, one item at a time for the Trainer."""

    def __init__(self, df):
        self.encodings = tokenizer(list(df["text"]), truncation=True, padding="max_length",
                                   max_length=MAX_LENGTH)
        self.labels = list(df["label_id"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: torch.tensor(values[i]) for key, values in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


train_ds, val_ds, test_ds = ClaimDataset(train_df), ClaimDataset(val_df), ClaimDataset(test_df)
hard_en_ds, hard_multi_ds = ClaimDataset(hard_en), ClaimDataset(hard_multi)

# See how MuRIL splits Hindi, Telugu and Hinglish text into tokens
for sample in ["Meri gaadi chori ho gayi", "मेरी गाड़ी चोरी हो गई", "నా కారు దొంగిలించబడింది"]:
    print(sample, "->", tokenizer.tokenize(sample))

In [ ]:
# Step 5: load MuRIL with a 6-class classification head, then add LoRA
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                                  # rank: size of the small LoRA matrices
    lora_alpha=16,                        # scaling of the LoRA update
    lora_dropout=0.1,                     # dropout to reduce overfitting
    target_modules=["query", "value"],    # BERT's attention query and value layers
    modules_to_save=["classifier"],       # the new classification head is trained fully
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()       # shows how few parameters LoRA trains

In [ ]:
# Step 6: train. The Trainer runs the training loop and checks validation after every epoch.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}


args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    learning_rate=5e-4,                   # LoRA uses a higher learning rate than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",                # check validation score after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,          # keep the epoch with the best validation F1
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    logging_steps=10,
    report_to="none",                     # no external logging service
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)

start = time.time()
trainer.train()
train_minutes = (time.time() - start) / 60
print(f"Training time: {train_minutes:.1f} minutes")

In [ ]:
# Step 7: evaluate on data the model never saw during training
def evaluate(dataset, df, name):
    output = trainer.predict(dataset)
    preds = np.argmax(output.predictions, axis=-1)
    acc = accuracy_score(df["label_id"], preds)
    f1 = f1_score(df["label_id"], preds, average="macro")
    print(f"\n===== {name}: accuracy {acc:.3f} | macro F1 {f1:.3f} =====")
    print(classification_report(df["label_id"], preds, labels=list(range(len(LABELS))),
                                target_names=LABELS, zero_division=0))
    return [id2label[p] for p in preds], acc, f1


_, test_acc, test_f1 = evaluate(test_ds, test_df, "Test set (same style as training)")
hard_en["fine_tuned_pred"], en_acc, en_f1 = evaluate(hard_en_ds, hard_en, "English hard set")
hard_multi["fine_tuned_pred"], multi_acc, multi_f1 = evaluate(hard_multi_ds, hard_multi, "Multilingual hard set")

# Both hard sets together (72 claims)
hard = pd.concat([hard_en, hard_multi], ignore_index=True)
hard_acc = accuracy_score(hard["label"], hard["fine_tuned_pred"])
hard_f1 = f1_score(hard["label"], hard["fine_tuned_pred"], average="macro")
print(f"All hard claims: accuracy {hard_acc:.3f} | macro F1 {hard_f1:.3f}")

# Accuracy per language on the hard claims
hard["ft_correct"] = hard["label"] == hard["fine_tuned_pred"]
print("\nAccuracy by language (hard claims):")
print(hard.groupby("language")["ft_correct"].agg(["mean", "count"]).round(3))

# Confusion matrix: rows = true type, columns = predicted type
cm = confusion_matrix(hard["label"], hard["fine_tuned_pred"], labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", colorbar=False)
plt.xticks(rotation=45)
plt.title("MuRIL + LoRA - all 72 hard claims")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# Show the mistakes, so we can explain them in the model card
errors = hard[~hard["ft_correct"]][["text", "language", "label", "fine_tuned_pred"]]
print(f"Mistakes: {len(errors)} of {len(hard)}")
errors

In [ ]:
# Step 8: a simple predict function (the FastAPI service will use the same logic)
model.eval()


def predict_claim_type(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    best = int(torch.argmax(probs))
    return id2label[best], round(float(probs[best]), 3)


for text in ["A lorry hit my car from behind at the signal and the bumper is broken",
             "Kal raat meri gaadi ghar ke bahar se chori ho gayi",
             "बारिश में मेरी कार पानी में डूब गई",
             "నా కారుకు మంటలు అంటుకున్నాయి"]:
    print(predict_claim_type(text), "<-", text)

# Confidence on every hard claim: used on Day 3 to choose the threshold for the GPT fallback
hard["ft_confidence"] = [predict_claim_type(t)[1] for t in hard["text"]]
print("\nAverage confidence when correct:", round(hard[hard.ft_correct]["ft_confidence"].mean(), 3))
print("Average confidence when wrong:  ", round(hard[~hard.ft_correct]["ft_confidence"].mean(), 3))

# Speed test: average time for one prediction
start = time.time()
for text in hard["text"]:
    predict_claim_type(text)
ft_ms = (time.time() - start) / len(hard) * 1000
print(f"\nFine-tuned model: {ft_ms:.1f} ms per claim on {device}")

In [ ]:
# Step 9: save the LoRA adapter (small file) + tokenizer + label map
SAVE_DIR = "claim_classifier_lora"
if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/labels.json", "w") as f:
    json.dump({"labels": LABELS, "base_model": MODEL_NAME, "max_length": MAX_LENGTH}, f, indent=2)

size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, n)) for n in os.listdir(SAVE_DIR)) / 1e6
print(f"Saved to {SAVE_DIR} ({size_mb:.1f} MB)")
print(os.listdir(SAVE_DIR))

## Step 10: compare with zero-shot GPT-4o-mini
Uses the `OPENAI_API_KEY` secret (key icon in the left sidebar, Notebook access on). Never paste a key into a cell.

Cost: 72 short calls, well under Rs 1.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# gpt-4o-mini prices in USD per 1 million tokens. Check https://openai.com/api/pricing and update if changed.
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60

PROMPT = """You classify motor insurance claim descriptions. They may be in English, Hindi, Telugu,
Hinglish or Tenglish. Reply with exactly one label from this list and nothing else:
collision, theft, flood, fire, vandalism, glass_only

Rules:
- Label the main cause of damage. An accident that also cracks the windshield is collision.
- Deliberate damage by people is vandalism, even if only glass broke.
- Burnt damage is fire, even if someone set it on purpose.
- glass_only only when nothing except glass is damaged.
- Stolen parts are theft."""


def gpt_classify(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "system", "content": PROMPT}, {"role": "user", "content": text}],
    )
    answer = response.choices[0].message.content.strip().lower()
    cost = (response.usage.prompt_tokens * PRICE_INPUT + response.usage.completion_tokens * PRICE_OUTPUT) / 1e6
    return answer, cost


gpt_preds, gpt_cost = [], 0.0
start = time.time()
for text in hard["text"]:
    label, cost = gpt_classify(text)
    gpt_preds.append(label if label in LABELS else "invalid")   # count invalid answers as wrong
    gpt_cost += cost
gpt_ms = (time.time() - start) / len(hard) * 1000

hard["gpt_pred"] = gpt_preds
hard["gpt_correct"] = hard["label"] == hard["gpt_pred"]
def acc_f1(df, col):
    return (round(accuracy_score(df["label"], df[col]), 3),
            round(f1_score(df["label"], df[col], labels=LABELS, average="macro", zero_division=0), 3))


gpt_acc, gpt_f1 = acc_f1(hard, "gpt_pred")
gpt_en_acc, _ = acc_f1(hard.iloc[:len(hard_en)], "gpt_pred")
gpt_multi_acc, _ = acc_f1(hard.iloc[len(hard_en):], "gpt_pred")
print(f"GPT-4o-mini on all 72 hard claims: accuracy {gpt_acc} | macro F1 {gpt_f1}")
print(f"Average time {gpt_ms:.0f} ms per claim | total cost ${gpt_cost:.5f}")
print("\nAccuracy by language (both models):")
print(hard.groupby("language")[["ft_correct", "gpt_correct"]].mean().round(3))

In [ ]:
# Step 11: side-by-side comparison on the same 72 hard claims
comparison = pd.DataFrame([
    {"model": "MuRIL + LoRA (fine-tuned)", "english_hard_acc": round(en_acc, 3),
     "multilingual_hard_acc": round(multi_acc, 3), "all_hard_acc": round(hard_acc, 3),
     "all_hard_macro_f1": round(hard_f1, 3), "ms_per_claim": round(ft_ms, 1), "usd_per_1000_claims": 0.0},
    {"model": "GPT-4o-mini (zero-shot prompt)", "english_hard_acc": gpt_en_acc,
     "multilingual_hard_acc": gpt_multi_acc, "all_hard_acc": gpt_acc, "all_hard_macro_f1": gpt_f1,
     "ms_per_claim": round(gpt_ms, 1), "usd_per_1000_claims": round(gpt_cost / len(hard) * 1000, 4)},
])
print(comparison.to_string(index=False))

results = {
    "version": "v2",
    "base_model": MODEL_NAME,
    "languages": ["en", "hi", "te", "hinglish", "tenglish"],
    "lora": {"r": 8, "alpha": 16, "dropout": 0.1, "target_modules": ["query", "value"]},
    "training": {"epochs": EPOCHS, "learning_rate": 5e-4, "train_rows": len(train_df), "val_rows": len(val_df),
                 "test_rows": len(test_df), "train_minutes": round(train_minutes, 2), "device": device},
    "test_set": {"accuracy": round(test_acc, 3), "macro_f1": round(test_f1, 3)},
    "english_hard_set": {"rows": len(hard_en), "accuracy": round(en_acc, 3), "macro_f1": round(en_f1, 3)},
    "multilingual_hard_set": {"rows": len(hard_multi), "accuracy": round(multi_acc, 3), "macro_f1": round(multi_f1, 3)},
    "all_hard": {"rows": len(hard), "accuracy": round(hard_acc, 3), "macro_f1": round(hard_f1, 3),
                 "ms_per_claim": round(ft_ms, 1)},
    "gpt_4o_mini_all_hard": {"accuracy": gpt_acc, "macro_f1": gpt_f1, "english_accuracy": gpt_en_acc,
                             "multilingual_accuracy": gpt_multi_acc, "ms_per_claim": round(gpt_ms, 1),
                             "total_cost_usd": round(gpt_cost, 5)},
    "adapter_size_mb": round(size_mb, 2),
    "v1_baseline": {"base_model": "distilbert-base-uncased", "english_only": True,
                    "test_accuracy": 0.967, "english_hard_accuracy": 0.861,
                    "note": "failed on Hinglish words (gaadi, paani) and unusual stolen parts"},
}
with open("classifier_results.json", "w") as f:
    json.dump(results, f, indent=2)
comparison.to_csv("model_comparison.csv", index=False)
hard[["text", "language", "label", "fine_tuned_pred", "ft_confidence", "gpt_pred"]].to_csv(
    "hard_test_predictions.csv", index=False)

In [ ]:
# Step 12: package everything in the same folders as the repo, then download one zip
if os.path.exists("day2_outputs"):
    shutil.rmtree("day2_outputs")
os.makedirs("day2_outputs/models", exist_ok=True)
os.makedirs("day2_outputs/eval", exist_ok=True)
shutil.copytree(SAVE_DIR, f"day2_outputs/models/{SAVE_DIR}", dirs_exist_ok=True)
for name in ["classifier_results.json", "model_comparison.csv", "hard_test_predictions.csv", "confusion_matrix.png"]:
    shutil.copy(name, f"day2_outputs/eval/{name}")
shutil.make_archive("day2_outputs_v2", "zip", "day2_outputs")

from google.colab import files
files.download("day2_outputs_v2.zip")